# 06 - Build a RAG pipeline

Everything comes together. You have all three pieces:

- an LLM you can prompt (notebook 02)
- a searchable vector database of chunks (notebooks 03 to 05)
- the idea of pasting retrieved text into the prompt (notebook 01)

This notebook wires them into a single ask() function: retrieve, augment,
generate, with sources.

**Before you run this notebook**: run notebook 05 at least once, so the
chroma_db folder exists with the Aurora Dynamics chunks in it.

**What you will learn**

- How to write the retrieve, augment, and generate steps as functions
- How to make the model say "I don't know" instead of hallucinating
- How to show sources for every answer

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

load_dotenv("../.env")

client = OpenAI()
MODEL = "gpt-4o-mini"

chroma_client = chromadb.PersistentClient(path="../chroma_db")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)
collection = chroma_client.get_or_create_collection(
    name="aurora_docs",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() == 0:
    raise RuntimeError("The collection is empty. Run notebook 05-vector-search.ipynb first.")
print("Chunks available:", collection.count())

Chunks available: 11


## Step 1: retrieve

Given a question, return the k most relevant chunks and their sources.
This is exactly the query from notebook 05, wrapped in a function.

In [2]:
def retrieve(question, k=4):
    results = collection.query(query_texts=[question], n_results=k)
    chunks = results["documents"][0]
    sources = [meta["source"] for meta in results["metadatas"][0]]
    return chunks, sources

chunks, sources = retrieve("How many vacation days do employees get?")
for source, chunk in zip(sources, chunks):
    print(f"[{source}] {chunk[:80]}...")

[03-leave-policy.txt] Aurora Dynamics - Employee Handbook: Leave Policy This policy applies to all ful...
[03-leave-policy.txt] public holiday calendar. In addition, the whole company closes for a shared wint...
[04-remote-work-policy.txt] Aurora Dynamics - Employee Handbook: Remote Work Policy Aurora Dynamics uses a h...
[04-remote-work-policy.txt] time budget for home office equipment. The budget is claimed through Compass wit...


## Step 2: augment

Build the prompt. This template is the heart of a RAG system, and it does
three jobs:

1. gives the model the retrieved chunks as context, labeled with sources
2. orders it to use only that context, which is the anti-hallucination rule
3. tells it exactly what to say when the context does not contain the answer

In [3]:
SYSTEM_PROMPT = """You answer questions about Aurora Dynamics using ONLY the
provided context. Rules:
- If the context does not contain the answer, reply exactly:
  "I don't know based on the available documents."
- Never use outside knowledge and never guess.
- Mention the source file(s) your answer came from."""

def build_prompt(question, chunks, sources):
    context = "\n\n".join(
        f"[source: {source}]\n{chunk}" for source, chunk in zip(sources, chunks)
    )
    return f"""Context:
{context}

Question: {question}"""

print(build_prompt("How many vacation days?", chunks[:1], sources[:1]))

Context:
[source: 03-leave-policy.txt]
Aurora Dynamics - Employee Handbook: Leave Policy This policy applies to all full time employees of Aurora Dynamics. Annual leave: Every employee receives 24 days of paid annual leave per calendar year. Unused days can be carried over, up to a maximum of 6 days, and carried days must be used before March 31 of the following year. Sick leave: Employees receive 12 days of paid sick leave per year. A doctor's note is required only for absences longer than 3 consecutive days. Parental leave: Primary caregivers receive 26 weeks of paid parental leave. Secondary caregivers receive 8 weeks. Leave can be taken any time within the first year after birth or adoption. Public holidays: Each office follows its local public holiday calendar. In addition, the whole company closes for a shared winter break from December 25 to January 1. Requesting leave: All leave is requested through the internal tool

Question: How many vacation days?


## Step 3: generate, and the full pipeline

The last step is the LLM call from notebook 02 with temperature 0. Chaining
all three steps gives us the complete RAG function - about 15 lines total.

In [4]:
def ask(question, k=4, show_sources=True):
    chunks, sources = retrieve(question, k)                  # 1. retrieve
    prompt = build_prompt(question, chunks, sources)         # 2. augment
    response = client.chat.completions.create(               # 3. generate
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    answer = response.choices[0].message.content
    if show_sources:
        answer += "\n\nRetrieved from: " + ", ".join(sorted(set(sources)))
    return answer

## Trying it out

The moment of truth. In notebook 01 the model had no idea about Aurora
Dynamics. Now:

In [5]:
print(ask("How many days of paid annual leave do employees get?"))

Employees receive 24 days of paid annual leave per calendar year. [source: 03-leave-policy.txt]

Retrieved from: 03-leave-policy.txt, 04-remote-work-policy.txt, 06-q1-2025-update.txt


In [6]:
print(ask("A robot stopped in a corridor and is blinking orange. What do I do?"))

To resolve the issue of the robot stopping in a corridor and blinking orange, you should wipe the lidar dome with the microfiber cloth from the maintenance kit. If the robot still blinks orange after cleaning, restart it by holding the power button for 8 seconds. If the issue repeats more than twice in one day, escalate to level 2 support. 

[source: 05-support-runbook.txt]

Retrieved from: 02-product-guide.txt, 05-support-runbook.txt


In [7]:
print(ask("How much does the Carrier X2 cost and what does the price include?"))

The Carrier X2 costs 2,900 dollars per robot per month, which includes maintenance, software updates, and 24/7 remote monitoring. 

[source: 02-product-guide.txt]

Retrieved from: 02-product-guide.txt, 06-q1-2025-update.txt


## The honesty test

Ask something the documents genuinely do not cover. A plain LLM would
happily improvise. Our system refuses, because the system prompt gives it a
safe exit:

In [8]:
print(ask("What color is the Carrier X2?"))

I don't know based on the available documents.

Retrieved from: 02-product-guide.txt, 05-support-runbook.txt, 06-q1-2025-update.txt


This behavior does not come for free from RAG itself. It comes from the
prompt rules. Delete the "reply exactly I don't know" rule and the model
starts guessing again. Prompt design is a real part of RAG engineering.

## A question that spans documents

Retrieval pulls the top k chunks regardless of which file they come from, so
one answer can combine several documents:

In [9]:
print(ask("Where does Aurora Dynamics have offices, and which one is hiring the most in 2025?"))

Aurora Dynamics has offices in Bengaluru, India (headquarters), Austin, Texas (engineering office), and Lisbon, Portugal (customer support hub). The Austin engineering office is hiring the most in 2025. 

(Source: 01-company-overview.txt, 06-q1-2025-update.txt)

Retrieved from: 01-company-overview.txt, 04-remote-work-policy.txt, 06-q1-2025-update.txt


## Exercise

1. Ask ask() three questions of your own about the documents in
   [data](../data/).
2. Call ask() with k=1. Find a question that gets a worse answer with k=1
   than with k=4, and reason about why.
3. Edit SYSTEM_PROMPT to remove the "I don't know" rule, re-run the honesty
   test, and watch what changes.

In [10]:
# Try the exercise here


## What you built

```
question -> retrieve (ChromaDB) -> augment (prompt template) -> generate (LLM) -> answer + sources
```

You wrote every step yourself, in plain Python, with no framework. That
means nothing in RAG is magic to you anymore. Notebook 07 shows the same
pipeline written with LangChain, so you can see what frameworks add and
decide when they are worth it.